# 🚀 Модуль 17. Урок 2. Продуктовый кейс: системная декомпозиция и дерево метрик

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_17_career/lesson_02_product_case_solving_framework.ipynb)

## 📌 Бизнес-контекст и методология
Классический продуктовый кейс: *«В марте выручка e-commerce упала на 12%. Руководство паникует. Ваши действия как ведущего аналитика?»*.
Антипаттерн — сразу строить случайные графики.
Профессиональный алгоритм:
1. **Clarification:** Уточнение горизонта, сезонности, изменения цен и методологии учета.
2. **Metric Decomposition:** $\Delta Revenue = \Delta Traffic \times CR \times AOV + \dots$
3. **Segmentation:** Срез по платформам (iOS, Android, Web), когортам и каналам.
4. **Root Cause:** Выделение изолированного драйвера проблемы.

In [ ]:
# 🚀 Универсальная инициализация окружения и данных
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
PRIMARY = "#1779ba"
SUCCESS = "#3adb76"
ALERT = "#cc4b37"
WARNING = "#ffae00"

# Моделирование показателей марта vs февраля
case_metrics = pd.DataFrame({
    'Период': ['Февраль 2027', 'Март 2027'],
    'Traffic (Users)': [100000, 102000],        # Трафик вырос на +2%
    'Conversion_Rate': [0.035, 0.0305],         # CR упал с 3.5% до 3.05%
    'AOV ($)': [120, 119.5]                    # Средний чек почти стабилен
})
case_metrics['Orders'] = (case_metrics['Traffic (Users)'] * case_metrics['Conversion_Rate']).round()
case_metrics['Revenue ($)'] = case_metrics['Orders'] * case_metrics['AOV ($)']

display(case_metrics)
rev_drop_pct = (case_metrics.loc[1, 'Revenue ($)'] - case_metrics.loc[0, 'Revenue ($)']) / case_metrics.loc[0, 'Revenue ($)'] * 100
print(f"Падение выручки: {rev_drop_pct:.2f}%")

## 🔍 Разбор механики: Факторный анализ (Waterfall Decomposition)
Разложим абсолютное изменение выручки ($\Delta Revenue$) на вклад 3 факторов:
$$\Delta Revenue = \Delta \text{Traffic} \cdot CR_0 \cdot AOV_0 + Traffic_1 \cdot \Delta CR \cdot AOV_0 + Traffic_1 \cdot CR_1 \cdot \Delta AOV$$

In [ ]:
# Расчет факторного влияния компонентов
T0, T1 = case_metrics['Traffic (Users)']
CR0, CR1 = case_metrics['Conversion_Rate']
AOV0, AOV1 = case_metrics['AOV ($)']
R0, R1 = case_metrics['Revenue ($)']

delta_traffic_effect = (T1 - T0) * CR0 * AOV0
delta_cr_effect = T1 * (CR1 - CR0) * AOV0
delta_aov_effect = T1 * CR1 * (AOV1 - AOV0)
total_delta = R1 - R0

factor_decomp = pd.DataFrame({
    'Фактор': ['1. Базовая выручка (Февраль)', 'Вклад трафика (+2%)', 'Вклад падения CR', 'Вклад чека AOV', '2. Итого выручка (Март)'],
    'Значение ($)': [R0, delta_traffic_effect, delta_cr_effect, delta_aov_effect, R1],
    'Тип': ['База', 'Плюс', 'Минус', 'Минус', 'Итог']
})
display(factor_decomp)

## 📊 Визуализация бизнес-кейса: Водопад факторов падения (Waterfall)
Построим каскадный график, наглядно показывающий, что именно обвал конверсии ($CR$) сжег всю выручку.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=110)

factors = factor_decomp['Фактор'].tolist()
vals = factor_decomp['Значение ($)'].tolist()
colors = [PRIMARY, SUCCESS, ALERT, WARNING, PRIMARY]

bars = ax.bar(range(len(factors)), vals, color=colors, edgecolor='#333333', linewidth=0.6, width=0.55)
ax.set_xticks(range(len(factors)))
ax.set_xticklabels(factors, fontsize=9.5, rotation=10, ha='right')
ax.set_ylabel("Выручка ($)", fontsize=11)
ax.set_title("Факторный анализ просадки выручки: Обвал конверсии как ключевой драйвер", fontsize=12, fontweight='bold', pad=15)

for b, v in zip(bars, vals):
    ax.annotate(f"${v:,.0f}", xy=(b.get_x() + b.get_width()/2, max(v, 0)),
                xytext=(0, 4 if v >= 0 else -14), textcoords="offset points",
                ha='center', va='bottom' if v >= 0 else 'top', fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.show()

### 🎯 Практическое задание 1: Доля падения от CR
Вычислите, какой процент от суммарного падения выручки обусловлен именно фактором конверсии.

In [ ]:
### ВАШ КОД ЗДЕСЬ
cr_share_pct = abs(delta_cr_effect) / (abs(delta_cr_effect) + abs(delta_aov_effect)) * 100
print(f"Доля негативного влияния CR среди факторов падения: {cr_share_pct:.1f}%")

assert cr_share_pct > 90, "Падение конверсии должно объяснять более 90% негативного тренда"
print("✅ Тест успешно пройден: факторный анализ точно локализовал корневую причину!")

### 💡 Вывод для C-Level на собеседовании
*«Проблема не в маркетинге — трафик вырос на 2%. Проблема внутри воронки: конверсия упала на 13% rel. Следующий шаг — сегментация воронки по шагам (Cart $\rightarrow$ Checkout $\rightarrow$ Payment) и платформам (iOS/Web), где скорее всего произошел технический сбой платежного шлюза»*.